# 04 · Modeling — classification end to end

**Dataset:** `data/titanic_clean.csv`.
**Covers:** baselines · model choice · cross-validation · metrics · tuning ·
over/underfitting · feature importance.
**Time yourself:** ~40 minutes.

The setup cell below does the feature work from notebooks 01–03 for you, so you can
spend the time on modeling.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

df = pd.read_csv('data/titanic_clean.csv')
df.columns = df.columns.str.lower()

# --- features (from notebooks 01-03) ---
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)
df['family_bin'] = pd.cut(df['family_size'], bins=[0, 1, 4, 20],
                          labels=['alone', 'small', 'large'])
df['title'] = df['name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
common = df['title'].value_counts()[lambda s: s >= 10].index
df['title'] = df['title'].where(df['title'].isin(common), 'Rare')
df['fare_per_person'] = df['fare'] / df['ticket'].map(df['ticket'].value_counts())
df['has_cabin'] = df['cabin'].notna().astype(int)

NUMERIC = ['pclass', 'age', 'fare_per_person', 'is_alone', 'has_cabin']
CATEGORICAL = ['sex', 'embarked', 'title', 'family_bin']
FEATURES = NUMERIC + CATEGORICAL

X = df[FEATURES]
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), NUMERIC),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                          ('encode', OneHotEncoder(handle_unknown='ignore',
                                                   drop='first'))]), CATEGORICAL),
    ])

print(X_train.shape, X_test.shape, '| survival rate:', round(y.mean(), 3))
X_train.head()

---

## Part A — Baselines

### Q1. Before any model: what accuracy does a model that always predicts the majority class
get? This is the number every result must beat.

<details><summary>hint</summary>

`DummyClassifier(strategy='most_frequent')`. Quoting this number unprompted is a strong signal in an interview.

</details>

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print('dummy accuracy:', dummy.score(X_test, y_test).round(3))

# ~0.62 just from always saying "died". Any model at 65% is barely doing anything.

### Q2. Fit a logistic regression in a pipeline. Report train and test accuracy.
Is it beating the dummy?

<details><summary>hint</summary>

Wrap `make_preprocessor()` and the classifier in a `Pipeline`. Use `max_iter=1000` — the default 100 won't converge and will warn at you.

</details>

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([('pre', make_preprocessor()),
                   ('clf', LogisticRegression(max_iter=1000, random_state=42))])
logreg.fit(X_train, y_train)

print('train:', logreg.score(X_train, y_train).round(3))
print('test :', logreg.score(X_test, y_test).round(3))

# ~0.83 test vs 0.62 dummy -- a real +21 points. And train/test are close, so it isn't
# overfitting. This is now the number a fancier model has to beat.

---

## Part B — Cross-validation

### Q3. A single split on 891 rows is noisy. Cross-validate the logistic regression with
5-fold stratified CV. Report mean **and** std. Why `StratifiedKFold` and not `KFold`?

<details><summary>hint</summary>

`cross_val_score(pipe, X_train, y_train, cv=cv)`. The std is what tells you whether a difference between two models is real.

</details>

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(logreg, X_train, y_train, cv=cv, scoring='accuracy')
print('fold scores:', scores.round(3))
print(f'CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}')

# Stratified keeps the ~38% survival rate in every fold. With plain KFold a fold can end
# up with a skewed class mix, which adds variance to the estimate for no reason.
# The std (~0.02-0.03) is the real point: two models 1 point apart are indistinguishable.

### Q4. Compare 5 model families with the same CV, in one loop: logistic regression, decision
tree, random forest, gradient boosting, and KNN. Report mean ± std for each, sorted.
Which wins — and is the win larger than the noise?

<details><summary>hint</summary>

Build the dict, loop, collect into a DataFrame. Then compare the gaps against the stds before you announce a winner.

</details>

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
}

results = []
for name, clf in models.items():
    pipe = Pipeline([('pre', make_preprocessor()), ('clf', clf)])
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    results.append({'model': name, 'mean': s.mean(), 'std': s.std()})

res = pd.DataFrame(results).sort_values('mean', ascending=False)
display(res.round(3))

# GradientBoosting ~0.837, LogisticRegression ~0.827, RandomForest ~0.820,
# DecisionTree ~0.802, KNN ~0.796.
# Read it honestly: the stds are ~0.013-0.032, so the top three are separated by less than
# one std. That is NOT a ranking -- on 891 rows those three are statistically tied, and
# re-running with a different random_state would reshuffle them. The only defensible
# statements are "boosting, logreg and RF are equivalent here" and "KNN and the untuned
# tree are behind". Note the plain logistic regression is holding its own against the
# ensembles -- on a small, mostly-linear problem it usually does.

---

## Part C — Metrics

### Q5. Accuracy hides things. Print a full classification report and a confusion matrix for
the logistic regression. Which class does it fail on, and in which direction?

<details><summary>hint</summary>

`classification_report` gives precision/recall/F1 per class. `ConfusionMatrixDisplay.from_estimator` plots it in one line.

</details>

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

y_pred = logreg.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['died', 'survived']))

ConfusionMatrixDisplay.from_estimator(logreg, X_test, y_test,
                                      display_labels=['died', 'survived'])
plt.show()

# Overall accuracy ~0.83 looks uniform, but it isn't: recall is 0.86 on 'died' and 0.77 on
# 'survived'. The model is biased toward the majority class -- it misses roughly a quarter
# of real survivors (false negatives). If this were a rescue-prioritisation model, that
# asymmetry is the whole story, and the single accuracy figure conceals it completely.

### Q6. Compute ROC-AUC and average precision (PR-AUC). Plot the ROC curve.
Why does AUC use `predict_proba` rather than `predict`?

<details><summary>hint</summary>

`predict_proba(X)[:, 1]` is the positive-class probability. AUC is threshold-independent — that's the whole point of it.

</details>

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay

y_prob = logreg.predict_proba(X_test)[:, 1]     # column 1 = P(survived)
print('ROC-AUC :', roc_auc_score(y_test, y_prob).round(3))
print('PR-AUC  :', average_precision_score(y_test, y_prob).round(3))

RocCurveDisplay.from_estimator(logreg, X_test, y_test)
plt.plot([0, 1], [0, 1], 'k--', label='chance')
plt.legend()
plt.show()

# predict() has already collapsed the probabilities at the 0.5 threshold. AUC measures
# ranking quality across EVERY threshold, so it needs the scores, not the hard labels.
# Read it as: the probability a random survivor is scored above a random non-survivor.

### Q7. The default threshold is 0.5, which nothing about your problem chose for you.
Suppose missing a survivor is 3× worse than a false alarm. Find the threshold that
maximises recall subject to precision staying above 0.6.

<details><summary>hint</summary>

`precision_recall_curve` returns arrays of precision/recall per threshold. Note `thr` is one shorter than the other two — that's why the `np.r_[thr, 1.0]`.

</details>

In [ ]:
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(y_test, y_prob)
pr = pd.DataFrame({'threshold': np.r_[thr, 1.0], 'precision': prec, 'recall': rec})

ok = pr[pr['precision'] >= 0.6]
best = ok.loc[ok['recall'].idxmax()]
print(best.round(3))

plt.plot(pr['threshold'], pr['precision'], label='precision')
plt.plot(pr['threshold'], pr['recall'], label='recall')
plt.axvline(best['threshold'], color='k', ls='--', label=f"chosen {best['threshold']:.2f}")
plt.xlabel('threshold'); plt.legend(); plt.show()

y_pred_tuned = (y_prob >= best['threshold']).astype(int)
print(classification_report(y_test, y_pred_tuned, target_names=['died', 'survived']))

# Lowering the threshold below 0.5 buys recall at the cost of precision. The threshold is
# a *business* decision, not a model one -- and it's free: no retraining needed.

### Q8. Conceptual, no code needed. Titanic is ~38% positive — mildly imbalanced. If instead
it were **0.5%** positive (say, fraud), which of accuracy / ROC-AUC / PR-AUC would
you report, and why?

<details><summary>hint</summary>

Write out the FPR formula and ask what happens to it when TN is 99.5% of the data. That's the whole argument.

</details>

In [ ]:
# Accuracy: useless. "Never fraud" scores 99.5% and catches nothing.
# ROC-AUC: still misleading. Its x-axis is FPR = FP / (FP + TN), and with 99.5% negatives
#   TN is enormous, so thousands of false positives barely move the FPR. The curve looks
#   great while an analyst drowns in false alarms.
# PR-AUC (average_precision): the honest one. Precision = TP / (TP + FP) has no TN term at
#   all, so it can't be flattered by the majority class. Its baseline is the positive rate
#   (0.005), so a PR-AUC of 0.3 is a 60x lift -- unimpressive-looking, actually excellent.
# Report PR-AUC, plus recall at a fixed review budget ("precision@top-100"), which is what
# the business actually operates on.

---

## Part D — Tuning

### Q9. Tune a `RandomForestClassifier` with `GridSearchCV` over `n_estimators`, `max_depth`
and `min_samples_leaf`. Report the best params and the best CV score.
Note the `clf__` prefix — why is it needed?

<details><summary>hint</summary>

Params for a pipeline step are named `<step_name>__<param>`. `n_jobs=-1` uses all cores.

</details>

In [ ]:
from sklearn.model_selection import GridSearchCV

rf_pipe = Pipeline([('pre', make_preprocessor()),
                    ('clf', RandomForestClassifier(random_state=42))])

params = {
    'clf__n_estimators': [100, 300],
    'clf__max_depth': [4, 6, 8, None],
    'clf__min_samples_leaf': [1, 3, 5],
}
grid = GridSearchCV(rf_pipe, params, cv=cv, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print('best params:', grid.best_params_)
print('best CV    :', round(grid.best_score_, 3))
print('test       :', round(grid.score(X_test, y_test), 3))

# The `clf__` prefix addresses the step named 'clf' inside the pipeline; a double
# underscore descends one level. `pre__num__impute__strategy` would reach the imputer.
# Note the preprocessor is refit inside every fold -- that's why tuning a pipeline
# is leak-free while tuning a pre-transformed matrix isn't.

### Q10. That grid was 24 combos × 5 folds = 120 fits. Show you know the alternative: run
`RandomizedSearchCV` over a *wider* space with a fixed budget of 20 candidates.
When would you prefer it?

<details><summary>hint</summary>

`randint(low, high)` from scipy.stats gives a sampling distribution instead of a fixed list. `n_iter` is your compute budget.

</details>

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

dists = {
    'clf__n_estimators': randint(100, 500),
    'clf__max_depth': randint(3, 15),
    'clf__min_samples_leaf': randint(1, 10),
    'clf__max_features': ['sqrt', 'log2', None],
}
search = RandomizedSearchCV(rf_pipe, dists, n_iter=20, cv=cv, scoring='accuracy',
                            random_state=42, n_jobs=-1)
search.fit(X_train, y_train)

print('best params:', search.best_params_)
print('best CV    :', round(search.best_score_, 3))

# Prefer random search when the space is large or continuous: grid cost explodes
# multiplicatively with each parameter, while random search's cost is whatever n_iter you
# set. It also spends its budget better -- if only 2 of 4 params actually matter, a grid
# wastes most of its fits re-testing the irrelevant ones at every level of the others.

---

## Part E — Diagnosis

### Q11. Deliberately overfit: fit an unconstrained decision tree and report train vs test.
Then fix it. Name the symptom and the cure.

<details><summary>hint</summary>

A decision tree with no `max_depth` will drive training accuracy toward 1.0. The gap between train and test is the diagnosis.

</details>

In [ ]:
tree = Pipeline([('pre', make_preprocessor()),
                 ('clf', DecisionTreeClassifier(random_state=42))])
tree.fit(X_train, y_train)
print('unconstrained -> train:', tree.score(X_train, y_train).round(3),
      '| test:', tree.score(X_test, y_test).round(3))

fixed = Pipeline([('pre', make_preprocessor()),
                  ('clf', DecisionTreeClassifier(max_depth=4, min_samples_leaf=10,
                                                 random_state=42))])
fixed.fit(X_train, y_train)
print('constrained   -> train:', fixed.score(X_train, y_train).round(3),
      '| test:', fixed.score(X_test, y_test).round(3))

# Symptom: train ~0.98, test ~0.78 -- a ~20 point gap. The tree grew until every leaf was
# pure, i.e. it memorised the training rows including their noise.
# Cure: constrain capacity (max_depth, min_samples_leaf), or average many trees
# (RandomForest). The constrained tree scores LOWER on train and HIGHER on test -- that
# trade is the entire point.

### Q12. Plot a learning curve for the tuned random forest. Does the model need **more data**
or **more capacity**? Justify from the shape of the curves.

<details><summary>hint</summary>

`learning_curve` returns `(sizes, train_scores, val_scores)` where the score arrays are (n_sizes, n_folds) — average across axis=1. Look at whether the validation curve is still climbing at the right-hand edge.

</details>

In [ ]:
from sklearn.model_selection import learning_curve

sizes, train_scores, val_scores = learning_curve(
    grid.best_estimator_, X_train, y_train, cv=cv,
    train_sizes=np.linspace(0.1, 1.0, 8), scoring='accuracy', n_jobs=-1)

plt.plot(sizes, train_scores.mean(axis=1), 'o-', label='train')
plt.plot(sizes, val_scores.mean(axis=1), 'o-', label='validation')
plt.fill_between(sizes, val_scores.mean(1) - val_scores.std(1),
                 val_scores.mean(1) + val_scores.std(1), alpha=0.15)
plt.xlabel('training examples'); plt.ylabel('accuracy'); plt.legend()
plt.title('Learning curve — tuned random forest')
plt.show()

# How to read one, in general:
#   - validation still CLIMBING at the right edge  -> more data is the cheapest win
#   - validation FLAT well before the edge          -> more rows of the same kind won't help
#   - large persistent GAP between train and val    -> overfitting: regularise or simplify
#   - both curves low and flat, converged together  -> underfitting: more capacity/features
# Here the validation curve rises steeply early then largely levels off, while train stays
# high -- so the remaining gap is variance, not a data shortage. More Titanic passengers
# wouldn't move it much; better features (or accepting ~0.83 as this data's ceiling) would.

---

## Part F — Interpretation

### Q13. Which features matter? Get the random forest's importances. Then get **permutation
importance** on the test set. Where do they disagree, and which do you trust?

<details><summary>hint</summary>

`best_rf.named_steps['clf'].feature_importances_` for Gini; `permutation_importance(model, X_test, y_test, n_repeats=20)` for the other. `get_feature_names_out()` on the preprocessor recovers the one-hot names.

</details>

In [ ]:
from sklearn.inspection import permutation_importance

best_rf = grid.best_estimator_
names = best_rf.named_steps['pre'].get_feature_names_out()

gini = pd.Series(best_rf.named_steps['clf'].feature_importances_,
                 index=names).sort_values(ascending=False)

perm = permutation_importance(best_rf, X_test, y_test, n_repeats=20,
                              random_state=42, scoring='accuracy')
perm_s = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
gini.head(8).plot.barh(ax=axes[0], title='Gini importance (train, one-hot level)')
perm_s.head(8).plot.barh(ax=axes[1], title='Permutation importance (test, raw column)')
for ax in axes:
    ax.invert_yaxis()
plt.tight_layout(); plt.show()

# Both agree sex/title dominate. They disagree because:
#  - Gini importance is computed on TRAINING data and is biased toward high-cardinality
#    and continuous features (fare_per_person, age get more possible split points), so it
#    inflates them even when they don't generalise.
#  - Permutation importance is measured on the TEST set against real score drop -- it
#    answers "if I destroy this column, how much worse do predictions get?"
# Trust permutation importance for "what actually drives performance". Its caveat: with
# two correlated features it shows both as unimportant, since permuting one leaves the
# other to carry the signal.

### Q14. Read the logistic regression's coefficients as odds ratios. Which feature has the
largest effect and what does its number literally mean?

The answer may not be the feature you expect — if so, explain why.

<details><summary>hint</summary>

`np.exp(coef)` converts a log-odds coefficient into an odds ratio. Before you interpret the winner, ask which *other* feature it's correlated with. Also remember the numeric features were standardised.

</details>

In [ ]:
lr_names = logreg.named_steps['pre'].get_feature_names_out()
coefs = pd.Series(logreg.named_steps['clf'].coef_[0], index=lr_names)

odds = pd.DataFrame({'coef': coefs, 'odds_ratio': np.exp(coefs)})
display(odds.reindex(coefs.abs().sort_values(ascending=False).index).round(3))

# The largest is `title_Mr` (coef ~ -1.97, odds ratio ~0.14), NOT `sex_male` -- which is a
# genuinely interesting result worth saying out loud.
# Literal reading: holding everything else fixed, having the title "Mr" multiplies the ODDS
# of survival by ~0.14, i.e. an ~86% reduction in odds. Odds, not probability.
#
# Why it beat sex_male: `title` and `sex` encode almost the same thing, so they are highly
# collinear and the model SPLITS the effect between them however it likes -- `title_Mr`
# absorbed most of the "adult male" signal and left `sex_male` with the remainder. This is
# exactly the coefficient instability that notebook 03's multicollinearity step warns
# about. The lesson: with correlated features, a coefficient is not "the importance of
# this feature" -- it's this feature's share of a jointly-explained effect, and that share
# is arbitrary. Don't read individual coefficients as importances unless the features are
# reasonably independent.
#
# Also note the numeric features went through StandardScaler, so their coefficients are
# "per 1 standard deviation", not per unit -- age's coefficient is per ~14 years, not per
# year. The one-hot columns are 0/1, so theirs are per-category-vs-baseline.

---

## Part G — Ship it

### Q15. Final checklist. Pick your final model, evaluate it **once** on the test set, compare
against the baseline, and save the whole pipeline to disk. Then state what you'd
monitor in production.

<details><summary>hint</summary>

`joblib.dump(pipeline, 'model.joblib')`. Save the *pipeline*, never the bare estimator.

</details>

In [ ]:
import joblib
from sklearn.metrics import classification_report

final = grid.best_estimator_
y_pred = final.predict(X_test)
y_prob = final.predict_proba(X_test)[:, 1]

print('dummy    :', dummy.score(X_test, y_test).round(3))
print('logreg   :', logreg.score(X_test, y_test).round(3))
print('final RF :', final.score(X_test, y_test).round(3))
print('ROC-AUC  :', roc_auc_score(y_test, y_prob).round(3), '\n')
print(classification_report(y_test, y_pred, target_names=['died', 'survived']))

joblib.dump(final, 'titanic_model.joblib')
print('saved -> titanic_model.joblib')

# The pipeline pickles preprocessing AND model together, so serving cannot drift out of
# sync with training. Saving a bare model and rebuilding the transforms by hand at serving
# time is the classic production bug.
#
# LOOK AT THE ACTUAL NUMBERS: dummy 0.615, logreg 0.827, tuned RF ~0.804. The tuned random
# forest won cross-validation (~0.840 vs 0.827) and then came in BELOW logistic regression
# on the held-out test set. That is not a bug, it's the lesson: selecting the best of ~24
# grid candidates on CV means the winner's CV score is partly luck, and that luck does not
# transfer. The CV score of a model you *chose using CV* is optimistically biased -- which
# is precisely why the test set has to stay untouched until this cell.
#
# So: ship the logistic regression. It matches or beats the RF on the only honest number
# here, trains in milliseconds, and produces coefficients you can defend to a regulator.
#
# Monitor in production: input drift (age/fare distributions vs training), prediction
# drift (share predicted positive), and score-vs-outcome once labels arrive. Alert on the
# input drift -- it's the leading indicator; accuracy decay is the lagging one.

---

## Debrief — the sentences worth having ready

- *"Baseline first."* Dummy 0.615 → logreg 0.827. That +21 points is the real result; the
  ensembles add nothing on top of it here.
- *"Is that gap bigger than the std?"* Almost always the right question when two CV
  numbers are compared. Here the top three models sit inside one std of each other — that
  is a tie, not a leaderboard.
- *"CV picked the RF; the test set disagreed."* Choosing the best of 24 candidates on CV
  bakes luck into the winner's score. It's why the test set stays sealed until the end.
- *"Accuracy is the wrong metric here because…"* — have the FPR argument ready.
- *"The threshold is a business decision."* Free performance, no retraining.
- *"Simplest model that hits the bar."* Being willing to *not* ship the fancy model reads
  as seniority.